# TripAdvisor Reviews – Debug & Test Notebook

Step-by-step testing for `src/sites/tripadvisor_reviews.py`.

Covers:
1. API call – fetching reviews (multi-language, pagination)
2. Ollama classification – topic & sentiment detection
3. Manual review input – adding reviews that the API can't fetch

**Requirements:** `TRIPADVISOR_API_KEY` env var, Ollama running locally with `mistral:7b`.

In [ ]:
from pathlib import Path
import sys, os, json

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

In [ ]:
from src.sites import tripadvisor_reviews as tr
import requests

api_key = os.getenv('TRIPADVISOR_API_KEY')
if not api_key:
    raise RuntimeError('Set TRIPADVISOR_API_KEY before running this notebook')

location_id = tr.LOCATION_IDS.get(tr.ANANEA_HOTEL)
print(f'Hotel: {tr.ANANEA_HOTEL}')
print(f'Location ID: {location_id}')
print(f'API key: {api_key[:8]}...')

## 1. API Call – Single Page (English)

In [ ]:
# Fetch a single page of English reviews
reviews_en, total_en = tr.ta_get_reviews_page(location_id, api_key, language='en', offset=0)
print(f'English reviews returned: {len(reviews_en)}')
print(f'Total English reviews available: {total_en}')
print()
for r in reviews_en:
    print(f"  [{r.get('id')}] {r.get('rating')}★ – {r.get('title')} ({r.get('published_date', '')[:10]})")

## 2. API Call – All Languages with Pagination

In [ ]:
# Fetch across all default languages with pagination
all_reviews = tr.ta_get_reviews(location_id, api_key)
print(f'Total unique reviews fetched: {len(all_reviews)}')
print()
for r in sorted(all_reviews, key=lambda x: x.get('published_date', ''), reverse=True):
    lang = r.get('_language', '?')
    print(f"  [{r.get('id')}] {r.get('rating')}★ lang={lang} – {r.get('title')} ({r.get('published_date', '')[:10]})")

In [ ]:
# Inspect raw API response for first review
if all_reviews:
    print(json.dumps(all_reviews[0], indent=2, ensure_ascii=False))

## 3. Ollama Classification – Test

In [ ]:
# Check Ollama availability
ollama_ok = tr.is_ollama_available()
print(f'Ollama available: {ollama_ok}')

if ollama_ok:
    # List available models
    resp = requests.get('http://localhost:11434/api/tags')
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'Models: {models}')

In [ ]:
# Classify a single review (pick the first with text)
test_review = next((r for r in all_reviews if r.get('text')), None)
if test_review and ollama_ok:
    print(f"Review: {test_review['title']}")
    print(f"Text: {test_review['text'][:300]}...")
    print()
    topics = tr.classify_review(test_review['text'])
    print(f'Classification result ({len(topics)} topics):')
    for t in topics:
        icon = '🟢' if t['sentiment'] == 'positive' else '🔴'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('No review with text found or Ollama not available')

In [ ]:
# Classify ALL fetched reviews and show results
if ollama_ok:
    for r in all_reviews:
        text = r.get('text', '')
        if not text:
            continue
        topics = tr.classify_review(text)
        r['topics'] = topics
        r['classified'] = True
        stars = '★' * r.get('rating', 0)
        topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
        print(f"{stars} {r.get('title', 'N/A')[:50]:50s} → {topic_str or '(none detected)'}")
else:
    print('Ollama not available – skip classification')

In [ ]:
# Debug: see the raw Ollama response for a specific review
# Change the index to test different reviews
DEBUG_INDEX = 0

if ollama_ok and all_reviews:
    review = all_reviews[DEBUG_INDEX]
    text = review.get('text', '')
    print(f"Review: {review.get('title')}")
    print(f"Full text:\n{text}\n")
    print('--- Sending to Ollama ---')
    
    # Make raw request to see full response
    prompt = f"""You are a hotel review analyst. Read the review below carefully and identify ALL topics mentioned, even briefly.

TOPICS (use these exact keys):
- employees: any mention of staff, service, friendliness, helpfulness, reception, concierge, team, waiters, management
- commodities: amenities, facilities, pool, gym, spa, room features, wifi, parking, fridge, toiletries, TV, air conditioning, balcony
- comfort: room comfort, bed quality, noise, quiet, space, temperature, room size, mattress, pillow, decor, ambiance
- cleaning: cleanliness, hygiene, tidiness, housekeeping, spotless, dirty, stains, towels changed, room serviced
- quality_price: value for money, pricing, worth, cost, overpriced, good deal, expensive, cheap, affordable, half board value
- meals: food, breakfast, restaurant, dining, bar, drinks, buffet, dinner, lunch, cuisine, menu, chef, kitchen, snacks
- return: whether the guest would return, come back, visit again, recommend, revisit, not return, wouldn't go back

RULES:
1. You MUST check each topic one by one. Go through the review sentence by sentence.
2. A single review CAN have both positive AND negative for the SAME topic.
3. Even brief or indirect mentions count (e.g. \"rooms were cleaned daily\" = cleaning positive).
4. If a topic is described positively, mark it positive. If negatively, mark it negative.
5. Output ONLY a JSON array. No explanation, no markdown.

Now analyze this review:
\"\"\"{text}\"\"\"

JSON array:"""
    
    payload = {'model': 'mistral:7b', 'prompt': prompt, 'stream': False,
               'options': {'temperature': 0.1, 'num_predict': 512}}
    resp = requests.post('http://localhost:11434/api/generate', json=payload, timeout=60)
    raw = resp.json().get('response', '')
    print(f'Raw Ollama response:\n{raw}')
    print()
    parsed = tr._parse_classification(raw)
    print(f'Parsed: {json.dumps(parsed, indent=2)}')

## 4. Manual Review Input

Since the API only returns 5 reviews per page, you can manually add reviews below.
Fill in the fields and run the cell to add them to the JSON file.

In [ ]:
from datetime import datetime
import hashlib

# ====== FILL IN YOUR REVIEW HERE ======
manual_review = {
    'reviewer_name': 'John D.',                    # reviewer name (used for ID)
    'rating': 5,                                    # 1-5
    'title': 'Amazing stay',                        # review title
    'text': 'Paste the full review text here...',   # full review text
    'published_date': '2026-03-01',                 # YYYY-MM-DD
    'trip_type': 'Couples',                         # Family, Couples, Solo, Business, Friends
}
# ========================================

# Generate unique ID from name + date + title
id_seed = f"{manual_review['reviewer_name']}_{manual_review['published_date']}_{manual_review['title']}"
review_id = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

# Format published_date to match API format
pub_date = manual_review['published_date']
if len(pub_date) == 10:  # YYYY-MM-DD
    pub_date = f"{pub_date}T00:00:00Z"

record = {
    'id': review_id,
    'hotel': tr.ANANEA_HOTEL,
    'location_id': location_id,
    'rating': manual_review['rating'],
    'title': manual_review['title'],
    'text': manual_review['text'],
    'published_date': pub_date,
    'travel_date': '',
    'trip_type': manual_review['trip_type'],
    'subratings': {},
    'helpful_votes': 0,
    'scraped_date': datetime.now().strftime('%Y-%m-%d'),
    'topics': [],
    'classified': False,
    'source': 'manual',
}

print(f'Generated ID: {review_id}')
print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
# Classify the manual review (optional)
if ollama_ok and record['text'] and record['text'] != 'Paste the full review text here...':
    topics = tr.classify_review(record['text'])
    record['topics'] = topics
    record['classified'] = True
    print(f'Classification ({len(topics)} topics):')
    for t in topics:
        icon = '🟢' if t['sentiment'] == 'positive' else '🔴'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('Ollama not available or no text – skipping classification')

In [ ]:
# Save the manual review to the JSON file
json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
existing = tr.load_reviews(json_path)

# Check for duplicates
existing_ids = {r['id'] for r in existing}
if record['id'] in existing_ids:
    print(f'⚠️  Review {record["id"]} already exists – skipping')
else:
    existing.append(record)
    tr.save_reviews(existing, json_path)
    print(f'✅ Saved! Total reviews: {len(existing)}')

## 5. Batch: Add Multiple Manual Reviews

Paste multiple reviews at once.

In [ ]:
# Add multiple reviews at once
batch_reviews = [
    {
        'reviewer_name': 'Maria S.',
        'rating': 4,
        'title': 'Good hotel, cold pool',
        'text': 'Replace with actual review text...',
        'published_date': '2026-02-15',
        'trip_type': 'Family',
    },
    # Add more reviews here...
]

json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
existing = tr.load_reviews(json_path)
existing_ids = {r['id'] for r in existing}
added = 0

for mr in batch_reviews:
    id_seed = f"{mr['reviewer_name']}_{mr['published_date']}_{mr['title']}"
    rid = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]
    
    if rid in existing_ids:
        print(f'⚠️  Skip duplicate: {mr["title"]}')
        continue
    
    pub_date = mr['published_date']
    if len(pub_date) == 10:
        pub_date = f"{pub_date}T00:00:00Z"
    
    rec = {
        'id': rid,
        'hotel': tr.ANANEA_HOTEL,
        'location_id': location_id,
        'rating': mr['rating'],
        'title': mr['title'],
        'text': mr['text'],
        'published_date': pub_date,
        'travel_date': '',
        'trip_type': mr.get('trip_type', ''),
        'subratings': {},
        'helpful_votes': 0,
        'scraped_date': datetime.now().strftime('%Y-%m-%d'),
        'topics': [],
        'classified': False,
        'source': 'manual',
    }
    
    # Classify if Ollama available
    if ollama_ok and rec['text'] and 'Replace with' not in rec['text']:
        try:
            topics = tr.classify_review(rec['text'])
            rec['topics'] = topics
            rec['classified'] = True
        except Exception as e:
            print(f'  Classification failed: {e}')
    
    existing.append(rec)
    existing_ids.add(rid)
    added += 1
    topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in rec.get('topics', []))
    print(f"✅ {mr['title']} → {topic_str or '(unclassified)'}")

if added:
    tr.save_reviews(existing, json_path)
    print(f'\nSaved {added} new reviews. Total: {len(existing)}')
else:
    print('No new reviews to add.')

## 6. Reclassify Reviews with Empty Topics

Finds reviews that were classified but got 0 topics (LLM missed), and retries.

In [ ]:
json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
reviews = tr.load_reviews(json_path)

# Find reviews that are "classified" but have 0 topics – likely LLM failures
needs_retry = [r for r in reviews if r.get('classified') and not r.get('topics') and r.get('text')]
# Also find unclassified reviews
unclassified = [r for r in reviews if not r.get('classified') and r.get('text')]

print(f'Total reviews: {len(reviews)}')
print(f'Classified with 0 topics (needs retry): {len(needs_retry)}')
print(f'Unclassified: {len(unclassified)}')

to_reclassify = needs_retry + unclassified

if ollama_ok and to_reclassify:
    for r in to_reclassify:
        try:
            topics = tr.classify_review(r['text'])
            r['topics'] = topics
            r['classified'] = True
            topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
            print(f"  ✅ {r.get('title', 'N/A')[:40]} → {topic_str or '(still none)'}")
        except Exception as e:
            print(f"  ❌ {r.get('title', 'N/A')[:40]} → Error: {e}")
    
    tr.save_reviews(reviews, json_path)
    print(f'\nDone. Saved {len(reviews)} reviews.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('Nothing to reclassify')

## 7. Current Data Summary

In [ ]:
import pandas as pd

json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
reviews = tr.load_reviews(json_path)

print(f'Total reviews: {len(reviews)}')
print(f'Classified: {sum(1 for r in reviews if r.get("classified"))}')
print(f'With topics: {sum(1 for r in reviews if r.get("topics"))}')
print(f'Manual: {sum(1 for r in reviews if r.get("source") == "manual")}')
print()

# Topic breakdown
topic_counts = {}
for r in reviews:
    for t in r.get('topics', []):
        key = f"{t['topic']} ({t['sentiment']})"
        topic_counts[key] = topic_counts.get(key, 0) + 1

if topic_counts:
    df = pd.DataFrame.from_dict(topic_counts, orient='index', columns=['Count'])
    df = df.sort_index()
    print('Topic sentiment counts:')
    print(df.to_string())
else:
    print('No topics classified yet.')